In [ ]:
# CADI Deep Learning - Semana 10
# # Semana 10:  Implementación de una Red Neuronal Siamesa para Reconocimiento Facial en Google Colab
# **Estudiante:** Jenifer Cárdenas Aguilera

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

# Cargamos números (MNIST) como equivalente a rostros
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

def crear_pares(x, y, n=5000):
    pares, etiquetas = [], []
    for i in range(n):
        # Par positivo (mismo número)
        idx1 = np.random.randint(len(x))
        label = y[idx1]
        idx2 = np.random.choice(np.where(y == label)[0])
        pares.append([x[idx1], x[idx2]])
        etiquetas.append(1)
        # Par negativo (número diferente)
        idx3 = np.random.choice(np.where(y != label)[0])
        pares.append([x[idx1], x[idx3]])
        etiquetas.append(0)
    return np.array(pares), np.array(etiquetas).astype("float32")

X_p, y_p = crear_pares(x_train, y_train)
print("Pares listos.")

In [ ]:
# La red que analiza cada imagen
base = models.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32)
])

input_l, input_r = layers.Input((28,28)), layers.Input((28,28))
feat_l, feat_r = base(input_l), base(input_r)

# Calculamos la diferencia (Distancia)
dist = layers.Lambda(lambda v: tf.abs(v[0] - v[1]))([feat_l, feat_r])
out = layers.Dense(1, activation="sigmoid")(dist)

model = models.Model(inputs=[input_l, input_r], outputs=out)
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

# Entrenamos rápido (3 épocas)
model.fit([X_p[:,0], X_p[:,1]], y_p, epochs=3, batch_size=64)

In [ ]:
# Probar con un par al azar del test
X_test_p, y_test_p = crear_pares(x_test, y_test, n=10)
p = model.predict([X_test_p[:1,0], X_test_p[:1,1]])[0][0]

plt.subplot(1,2,1); plt.imshow(X_test_p[0,0], cmap="gray")
plt.subplot(1,2,2); plt.imshow(X_test_p[0,1], cmap="gray")
plt.title(f"Similitud: {p:.2f} ({'Iguales' if p > 0.5 else 'Distintos'})")
plt.show()

## Análisis y Conclusiones: Redes Siamesas
### ¿Qué es esto?
A diferencia de las redes neuronales tradicionales que intentan clasificar una imagen (por ejemplo, decidir si "esto es un 5"), una Red Siamesa consiste en dos redes idénticas que comparten el mismo "cerebro" (sus pesos y parámetros son los mismos).

Su función no es etiquetar, sino comparar. El modelo procesa dos entradas simultáneamente y nos dice: "estas dos imágenes se parecen tanto que deben ser lo mismo".

## Análisis y Conclusiones Técnicas
### 1. Lógica de decisión
El modelo no se limita a memorizar categorías fijas; su propósito es aprender a proyectar imágenes en un espacio matemático o latente. En este espacio, el sistema utiliza funciones de distancia o similitud para la toma de decisiones, buscando reducir la distancia entre elementos que pertenecen a la misma clase y aumentarla entre los que son diferentes.  
### 2. Ventaja principal
La gran potencia de esta arquitectura es que permite el reconocimiento con pocos datos (One-shot o Few-shot learning). Una vez que la red entiende la lógica de comparación entre pares de datos, es capaz de comparar y reconocer rostros o patrones nuevos que nunca ha visto antes en su entrenamiento inicial.  
### 3. Uso en la vida real
Esta tecnología es fundamental para escenarios de reconocimiento facial y verificación de identidad. El sistema no necesita saber "quién" eres en términos generales; simplemente utiliza un mecanismo de comparación basado en distancia para validar tu cara actual contra la imagen de referencia guardada previamente.  